# Session - 15

# Advanced Data Transformation & Type Engineering with Pandas

In [1]:
import pandas as pd 
import numpy as np

# Task 1: The Code Translator (Mapping with .map())

In [2]:
# Loading Dataset
url = "Orders.csv"
df = pd.read_csv("Orders.csv",encoding="latin1")

df.head()

,Row ID,Order Priority,Discount,Unit Price,Shipping Cost,Customer ID,Customer Name,Ship Mode,Customer Segment,Product Category,...,Region,State or Province,City,Postal Code,Order Date,Ship Date,Profit,Quantity ordered new,Sales,Order ID
0,18606,Not Specified,0.01,2.88,0.50,2,Janice Fletcher,Regular Air,Corporate,Office Supplies,...,Central,Illinois,Addison,60101,5/28/2012,5/30/2012,1.3200,2,5.90,88525
1,20847,High,0.01,2.84,0.93,3,Bonnie Potter,Express Air,Corporate,Office Supplies,...,West,Washington,Anacortes,98221,7/7/2010,7/8/2010,4.5600,4,13.01,88522
2,23086,Not Specified,0.03,6.68,6.15,3,Bonnie Potter,Express Air,Corporate,Office Supplies,...,West,Washington,Anacortes,98221,7/27/2011,7/28/2011,-47.6400,7,49.92,88523
3,23087,Not Specified,0.01,5.68,3.60,3,Bonnie Potter,Regular Air,Corporate,Office Supplies,...,West,Washington,Anacortes,98221,7/27/2011,7/28/2011,-30.5100,7,41.64,88523
4,23088,Not Specified,0.00,205.99,2.50,3,Bonnie Potter,Express Air,Corporate,Technology,...,West,Washington,Anacortes,98221,7/27/2011,7/27/2011,998.2023,8,1446.67,88523


In [4]:
# Convert Order_Status_Code into Order_Status
df["Order_Status_Code"] = ["P", "S", "D", "S", "R"] * (len(df) // 5) + ["P", "S", "D", "S", "R"][:len(df) % 5]

df[["Order_Status_Code"]].head(10)

status_map = {
    "P": "Pending",
    "S": "Shipped",
    "D": "Delivered",
    "R": "Returned"
}

df["Order_Status"] = df["Order_Status_Code"].map(status_map)

df[["Order_Status_Code", "Order_Status"]].head(10)

,Order_Status_Code,Order_Status
0,P,Pending
1,S,Shipped
2,D,Delivered
3,S,Shipped
4,R,Returned
5,P,Pending
6,S,Shipped
7,D,Delivered
8,S,Shipped
9,R,Returned


In [5]:
# Identify any unmapped codes
unmapped_codes = df.loc[
    df["Order_Status"].isna(),
    "Order_Status_Code"
].unique()

unmapped_codes

<StringArray>
[]
Length: 0, dtype: str

In [6]:
df[["Order_Status_Code", "Order_Status"]].head(10)

,Order_Status_Code,Order_Status
0,P,Pending
1,S,Shipped
2,D,Delivered
3,S,Shipped
4,R,Returned
5,P,Pending
6,S,Shipped
7,D,Delivered
8,S,Shipped
9,R,Returned


# Why `.map()` is preferable to manual replacements

The `.map()` function is preferable because it provides a clear and efficient way to convert coded values into meaningful labels using a predefined dictionary. It is easier to maintain, reduces repetitive code, and makes the transformation consistent across the dataset.

# Task 2: The Rule Engine (Row-Level Logic with .apply())

In [7]:
def categorize_order(amount):
    if amount >= 100000:
        return "High Value"
    elif amount >= 50000:
        return "Medium Value"
    else:
        return "Low Value"

In [8]:
df["Order_Category"] = df["Sales"].apply(categorize_order)

df[["Sales", "Order_Category"]].head(10)

,Sales,Order_Category
0,5.90,Low Value
1,13.01,Low Value
2,49.92,Low Value
3,41.64,Low Value
4,1446.67,Low Value
5,2011.67,Low Value
6,1451.37,Low Value
7,6362.85,Low Value
8,113.25,Low Value
9,1515.17,Low Value


# Row-wise vs Column-wise apply

Row-wise `.apply(axis=1)` applies a function to each row. It is useful when the calculation depends on multiple columns in the same row.

Column-wise `.apply(axis=0)` applies a function to each column. It is useful when we want to perform an operation separately on each column.

In this task, we used `.apply()` on the `Sales` column to apply the pricing rule to each value.

# Task 3: The Type Forensics Unit (Type Conversion with astype())

In [9]:
# Convert Customer ID to string
df["Customer ID"] = df["Customer ID"].astype(str)

# Convert Quantity to integer
df["Quantity ordered new"] = df["Quantity ordered new"].astype(int)

# Convert Discount to float
df["Discount"] = df["Discount"].astype(float)

# Verify the data types
print("Updated Data Types:")
print(df[["Customer ID", "Quantity ordered new", "Discount"]].dtypes)

Updated Data Types:
Customer ID                 str
Quantity ordered new      int64
Discount                float64
dtype: object


# Consequences of Wrong Data Types

Incorrect data types can cause calculation errors and inaccurate analysis. For example, if a numeric column is stored as text, mathematical calculations may not work correctly. Incorrect data types can also affect sorting, comparisons, filtering, and data processing. Therefore, converting columns to the correct data type is important for reliable analysis and reporting.

# Task 4: The Error Handler (pd.to_numeric(errors='coerce'))

In [13]:
# Create the corrupted financial data
revenue_df = pd.DataFrame({
    'Revenue': [100000, 85000, 'N/A', 'missing', 72000]
})

# Convert Revenue to numeric
revenue_df['Revenue'] = pd.to_numeric(
    revenue_df['Revenue'],
    errors='coerce'
)

# Display the converted Revenue column
print("Converted Revenue column:")
print(revenue_df['Revenue'])

# Count how many values became NaN
nan_count = revenue_df['Revenue'].isna().sum()

print("\nNumber of values converted to NaN:", nan_count)

Converted Revenue column:
0    100000.0
1     85000.0
2         NaN
3         NaN
4     72000.0
Name: Revenue, dtype: float64

Number of values converted to NaN: 2


# Explain why coercion is safer than crashing.

The `pd.to_numeric()` function converts the Revenue values into numeric data.

The `errors='coerce'` parameter converts invalid values such as `N/A` and `missing` into `NaN` instead of stopping the program with an error.

In this example, 2 values were converted to `NaN` because they could not be converted into numbers.

Coercion is safer than crashing because the data-processing process can continue even when corrupted or invalid values are present. These invalid values can then be identified and handled separately.

# Task 5: The Transformation Pipeline (Integrated Scenario)

In [14]:
# 1. Map Order Status Codes
status_mapping = {
    'P': 'Pending',
    'S': 'Shipped',
    'D': 'Delivered',
    'R': 'Returned'
}

df['Order_Status'] = df['Order_Status_Code'].map(status_mapping)


# 2. Coerce Sales to numeric before creating categories
# Invalid/corrupted values will become NaN
df['Sales'] = pd.to_numeric(df['Sales'], errors='coerce')


# 3. Create Order Category using apply
def categorize_order(sales):
    if pd.isna(sales):
        return 'Unknown'
    elif sales >= 100000:
        return 'High'
    elif sales >= 50000:
        return 'Medium'
    else:
        return 'Low'

df['Order_Category'] = df['Sales'].apply(categorize_order)


# 4. Convert numeric columns to correct data types
df['Customer ID'] = df['Customer ID'].astype(str)
df['Quantity ordered new'] = pd.to_numeric(
    df['Quantity ordered new'], errors='coerce'
).astype('Int64')
df['Discount'] = pd.to_numeric(
    df['Discount'], errors='coerce'
)


# 5. Display the transformed DataFrame
print("Transformed DataFrame:")
display(df)

print("\nData Types:")
print(df.dtypes)

Transformed DataFrame:


,Row ID,Order Priority,Discount,Unit Price,Shipping Cost,Customer ID,Customer Name,Ship Mode,Customer Segment,Product Category,...,Postal Code,Order Date,Ship Date,Profit,Quantity ordered new,Sales,Order ID,Order_Status_Code,Order_Status,Order_Category
0,18606,Not Specified,0.01,2.88,0.50,2,Janice Fletcher,Regular Air,Corporate,Office Supplies,...,60101,5/28/2012,5/30/2012,1.320000,2,5.90,88525,P,Pending,Low
1,20847,High,0.01,2.84,0.93,3,Bonnie Potter,Express Air,Corporate,Office Supplies,...,98221,7/7/2010,7/8/2010,4.560000,4,13.01,88522,S,Shipped,Low
2,23086,Not Specified,0.03,6.68,6.15,3,Bonnie Potter,Express Air,Corporate,Office Supplies,...,98221,7/27/2011,7/28/2011,-47.640000,7,49.92,88523,D,Delivered,Low
3,23087,Not Specified,0.01,5.68,3.60,3,Bonnie Potter,Regular Air,Corporate,Office Supplies,...,98221,7/27/2011,7/28/2011,-30.510000,7,41.64,88523,S,Shipped,Low
4,23088,Not Specified,0.00,205.99,2.50,3,Bonnie Potter,Express Air,Corporate,Technology,...,98221,7/27/2011,7/27/2011,998.202300,8,1446.67,88523,R,Returned,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9421,20275,Critical,0.06,35.89,14.72,3402,Frederick Cole,Regular Air,Consumer,Office Supplies,...,25314,5/14/2013,5/15/2013,137.860000,13,447.87,87532,S,Shipped,Low
9422,20276,Critical,0.00,3.34,7.49,3402,Frederick Cole,Regular Air,Consumer,Office Supplies,...,25314,5/14/2013,5/14/2013,-39.070000,3,13.23,87532,D,Delivered,Low
9423,24491,Not Specified,0.08,550.98,45.70,3402,Frederick Cole,Delivery Truck,Consumer,Furniture,...,25314,9/12/2013,9/14/2013,-1225.029097,4,2215.93,87533,S,Shipped,Low
9424,25914,High,0.10,105.98,13.99,3403,Tammy Buckley,Express Air,Consumer,Furniture,...,82001,2/8/2010,2/11/2010,349.485000,5,506.50,87530,R,Returned,Low



Data Types:
Row ID                    int64
Order Priority              str
Discount                float64
Unit Price              float64
Shipping Cost           float64
Customer ID                 str
Customer Name               str
Ship Mode                   str
Customer Segment            str
Product Category            str
Product Sub-Category        str
Product Container           str
Product Name                str
Product Base Margin     float64
Region                      str
State or Province           str
City                        str
Postal Code               int64
Order Date                  str
Ship Date                   str
Profit                  float64
Quantity ordered new      Int64
Sales                   float64
Order ID                  int64
Order_Status_Code           str
Order_Status                str
Order_Category              str
dtype: object


# Explanation transformation order.

The transformation pipeline prepares the dataset for use in a data warehouse.

First, the Order_Status_Code values are mapped to meaningful status names such as Pending, Shipped, Delivered, and Returned.

Next, the Sales column is converted to numeric values using `errors='coerce'`. This converts corrupted or invalid values into `NaN` instead of causing the program to crash.

The Order_Category column is then created using `apply()`. Sales values of 100000 or more are classified as High, values from 50000 to below 100000 are classified as Medium, and values below 50000 are classified as Low. Missing Sales values are classified as Unknown.

Finally, the columns are converted to appropriate data types: Customer ID to string, Quantity ordered new to integer, and Discount to float.

The transformation order is important because corrupted numeric values should be handled before performing numerical comparisons. This makes the pipeline safer and produces a clean, consistent, warehouse-ready dataset.